<div align="center">
  <img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Travel%20and%20places/Rocket.png" width="120" />
  <h1 style="color: #1D9E75; font-size: 3.5em; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">💰 The Churn Crusher 💰</h1>
  <h3 style="color: #E85D24;">Predicting Telecom Customer Exit with Machine Learning</h3>
  <img src="https://capsule-render.vercel.app/api?type=waving&color=gradient&height=120&section=header" width="100%" />
</div>

### 🕵️ Why are we here?
In the world of telecom, keeping a customer is **5x cheaper** than finding a new one. When a customer leaves (churns), it's not just a lost subscription—it's a blow to the heart of the business! 

**Our Mission:** Use data to sniff out the churners before they start packing their bags! 🧳🏃‍♂️

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Objects/Hammer%20and%20Wrench.png" width="50" />
## Step 1: Loading the Heavy Artillery
Before we fight churn, we need our tools. We're bringing in the big guns: `Pandas` for data wrangling, `Seaborn` for gorgeous visuals, and `XGBoost` for the heavy lifting.

> *"A data scientist without libraries is like a chef without salt. Technical, but bland."* - Anonymous Wise Person.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="viridis")
print("✅ All systems go! Libraries deployed.")

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Objects/File%20Folder.png" width="50" />
## Step 2: Unboxing the Data
We use the **Telco Customer Churn** dataset. It contains 7,043 rows of customer behavior. Let's see what we're working with.

In [ ]:
df = pd.read_csv("churn.csv")

print(f"📂 Dataset comprises {df.shape[0]} customers across {df.shape[1]} features.")
print("\n✨ Peek at the data:")
df.head().style.background_gradient(cmap='YlGnBu')

### 🔍 Vital Stats Check
Are there missing values? Is the data imbalanced? (Spoiler: It always is). 

In [ ]:
print("📄 Detailed Checkup:")
display(df.describe().T)

churn_counts = df['Churn'].value_counts(normalize=True) * 100
print(f"\n⚖️ Churn Status: No ({churn_counts['No']:.1f}%) vs Yes ({churn_counts['Yes']:.1f}%)")
print("⚠️ Notice the imbalance! This is why we'll need SMOTE later!")

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Objects/Magnifying%20Glass%20Tilted%20Right.png" width="50" />
## Step 3: Visual Forensics (EDA)
Time to play detective! 🕵️‍♂️ We want to know: **Who is leaving?** 

Is it the monthly payers? The high-speed fiber users? Let's visualize the pain points.

### 📊 Churn Count: The Big Picture
How many people are actually walking out the door?

In [ ]:
plt.figure(figsize=(6,5))
sns.countplot(data=df, x='Churn', palette=['#1D9E75','#E85D24'])
plt.title("The Exit Count", fontsize=15, fontweight='bold')
plt.show()

### 📉 Feature vs Churn: The Trio of Truth
We examine `tenure`, `MonthlyCharges`, and `TotalCharges`. 

*   **Tenure:** Longer tenure usually means loyalty. 
*   **Monthly Charges:** High bills = Unhappy wallets. 
*   **Total Charges:** Cumulative spend.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

features = ['tenure', 'MonthlyCharges', 'TotalCharges']
for i, col in enumerate(features):
    sns.kdeplot(data=df, x=col, hue='Churn', shade=True, ax=axes[i], palette=['#1D9E75','#E85D24'])
    axes[i].set_title(f'Distribution of {col} by Churn', fontweight='bold')

plt.tight_layout()
plt.show()
print("💡 Insight: Churned customers (Orange) are concentrated at low tenure and high monthly charges!" )

### 📜 The Contract Curse
Check out how contract types affect loyalty. Monthly contracts are like bad relationships—quick to end!

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(data=df, x='Contract', hue='Churn', multiple='stack', shrink=.8, palette=['#1D9E75','#E85D24'])
plt.title("Churn by Contract Type: The 'Month-to-Month' Trap", fontsize=14, fontweight='bold')
plt.show()

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Objects/Soap.png" width="50" />
## Step 4: Data Scrubbing & Prep
Cleaning data is 80% of the job, and mostly involves making sure computers don't get confused by words like "Yes", "No", and "Internet Service".

**Plan:**
1. Drop the `customerID` (doesn't help predict anything).
2. Fill empty `TotalCharges` (the silent model killer).
3. Encode categories into numbers (machine language).

In [ ]:
df.drop('customerID', axis=1, inplace=True)
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Binary Mapping
mapping = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}
cols_to_map = ['Partner','Dependents','PhoneService','PaperlessBilling','Churn', 'gender']

for col in cols_to_map:
    if col in df.columns:
        df[col] = df[col].map(mapping).fillna(df[col])

print("✨ Basics scrubbed. Now for the One-Hot magic...")

In [ ]:
cat_cols = ['MultipleLines','InternetService','OnlineSecurity', 
            'OnlineBackup','DeviceProtection','TechSupport', 
            'StreamingTV','StreamingMovies','Contract','PaymentMethod']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f"🚀 Dataset expanded to {df.shape[1]} columns. Machines can now read our data!")

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Symbols/Triangular%20Ruler.png" width="50" />
## Step 5: The SMOTE Wizardry 🧙‍♂️
Since only ~26% of our data represents churners, a model could just guess "Normal" for everyone and be 74% accurate. That's a fail! 

**SMOTE** (Synthetic Minority Over-sampling Technique) creates "fake" churners by looking at the neighbors of real ones. It's like invited specialized actors to a party to balance the vibe!

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"⚖️ Training set balanced!")
print(f"Before SMOTE: Churn={y_train.sum()}, No Churn={len(y_train)-y_train.sum()}")
print(f"After SMOTE: Churn={y_train_res.sum()}, No Churn={len(y_train_res)-y_train_res.sum()}")

### 📏 Scaling for Fairness
Logistic Regression treats all numbers equally. If `TotalCharges` is 1000 and `gender` is 1, it might think charges are 1000x more important. Scaling brings everyone to the same playing field (0 to 1 range approx).

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_res)
X_test_sc = scaler.transform(X_test)
print("✅ All features measured on the same yardstick.")

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Activities/Trophy.png" width="50" />
## Step 6: The Model Showdown
We battle three titans of classification:
1. **Logistic Regression:** The Classic Statistician. 🏛️
2. **Random Forest:** The Wisdom of the Crowd (Trees). 🌲🌲🌲
3. **XGBoost:** The Modern Speed Demon. 🏎️💨

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = {}
for name, model in models.items():
    # Standardize data usage: LR needs scaled data, others don't strictly require it but it's safe.
    train_x = X_train_sc if name == "Logistic Regression" else X_train_res
    test_x = X_test_sc if name == "Logistic Regression" else X_test
    
    model.fit(train_x, y_train_res)
    preds = model.predict(test_x)
    proba = model.predict_proba(test_x)[:,1]
    
    auc = roc_auc_score(y_test, proba)
    results[name] = {"preds": preds, "proba": proba, "auc": auc, "model": model}
    print(f"🎯 {name} finished! AUC-ROC Score: {auc:.4f}")

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Objects/Scroll.png" width="50" />
## Step 7: The Verdict (Evaluation)
Which model is our champion? We look at the **ROC Curve** and the **Confusion Matrix**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['preds'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Stay', 'Churn'])
    disp.plot(ax=axes[i], cmap='YlGn', colorbar=False)
    axes[i].set_title(f"{name}\nAccuracy: {(np.sum(np.diag(cm))/np.sum(cm)):.2%}", fontweight='bold')

plt.tight_layout()
plt.show()

### 🏆 The ROC Curve Comparison
The closer the curve is to the top-left corner, the better. It shows the trade-off between sensitivity and specificity.

In [ ]:
plt.figure(figsize=(10, 7))
colors = ['#378ADD', '#1D9E75', '#E85D24']
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", lw=3, color=color)

plt.plot([0,1], [0,1], 'k--', alpha=0.5)
plt.xlabel("False Positive Rate (False Alarms)")
plt.ylabel("True Positive Rate (Caught Churners)")
plt.title("The Battle of ROC Curves", fontsize=15, fontweight='bold')
plt.legend()
plt.show()

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Smilies/Star-Struck.png" width="50" />
## Step 8: The Crystal Ball (Feature Importance)
Model built? Check. Verified? Check. Now, **tell us WHY!** 

What are the top secrets our model found for customer exit?

In [ ]:
rf = results['Random Forest']['model']
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 8))
sns.barplot(x=importances, y=importances.index, palette='magma')
plt.title("The Top 15 Churn Drivers", fontsize=16, fontweight='bold')
plt.xlabel("Importance Score")
plt.show()

print("🎊 BUSINESS RECOMMENDATION: Focus on customers with high fiber charges and month-to-month contracts. High risk detected!")

<div align="center">
  <img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Handshakes/Handshake.png" width="100" />
  <h2 style="color: #1D9E75;">Mission Accomplished!</h2>
  <p>We've cleaned, explored, balanced, and modeled our way to success.</p>
  <p><strong>Ready to save the business some millions? Let's go!</strong></p>
  <img src="https://capsule-render.vercel.app/api?type=waving&color=gradient&height=120&section=footer" width="100%" />
</div>